# Genotype–Phenotype Heterogeneity and Heterogeneity-Driven Infertility Management in Non-Classical 21-Hydroxylase Deficiency: A Clinical Analysis and Literature Review Exploration with `mlcroissant`
This notebook provides a template and example workflow for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by the Croissant schema at:
`https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json`


In [ ]:
# Ensure mlcroissant library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL (FAIR^2 dataset)
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata as a single object
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Each record set and field in the dataset is referenced by its unique `@id`. The dataset schema includes multiple record sets:

- Table 1: Clinical features
- Table 2: Hormone levels and imaging
- Table 3: Genetic variants

Let's list out their `@id`s and inspect their fields.

In [ ]:
# List available record sets (referenced by @id)
record_sets = list(dataset.record_sets.keys())
print("Record Sets IDs Available:")
for rs_id in record_sets:
    print(f"- {rs_id}")

# Show field IDs for each record set
for rs_id in record_sets:
    print(f"\nFields for RecordSet @id={rs_id}:")
    fields = dataset.record_sets[rs_id].fields
    for field in fields:
        print(f"  - Field @id: {field['@id']}, name: {field.get('name', '')}, type: {field.get('dataType', '')}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. All extraction uses `@id` references.

In [ ]:
# Extract records from each record set
dataframes = {}

for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"\nFirst 3 rows from RecordSet @id={rs_id}:")
    print(df.head(3))
    print(f"Fields (columns): {df.columns.tolist()}")

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps: filtering, normalization, grouping.

As an example, let us select a numeric field from Table 1 (clinical features)—for instance, 'age_at_diagnosis'—and filter patients older than a threshold. All fields are referenced by their `@id`.

Then, we normalize values and optionally group by another field such as 'infertility'.

In [ ]:
# Example using Table 1's RecordSet @id
clinical_rs_id = None
# Find Table 1 record set
for rs_id in record_sets:
    if 'clinical' in rs_id.lower() or 'table1' in rs_id.lower():
        clinical_rs_id = rs_id
        break
if clinical_rs_id is None:
    clinical_rs_id = record_sets[0]
    print(f"Defaulted clinical_rs_id to first RecordSet @id={clinical_rs_id}")

clinical_df = dataframes[clinical_rs_id]

# Find numeric field id (e.g., 'age_at_diagnosis')
numeric_field_id = None
for col in clinical_df.columns:
    if 'age' in col.lower():
        numeric_field_id = col
        break
if numeric_field_id is None:
    numeric_field_id = clinical_df.columns[0]

threshold = 25
filtered_df = clinical_df[clinical_df[numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouped by categorical field (e.g., 'infertility')
group_field_id = None
for col in clinical_df.columns:
    if 'infertility' in col.lower():
        group_field_id = col
        break
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"Grouped data by {group_field_id} (mean {numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of age_at_diagnosis (or the selected numeric field)
plt.figure(figsize=(6,4))
sns.histplot(clinical_df[numeric_field_id], bins=5, kde=True)
plt.title(f"Distribution of {numeric_field_id} in Table 1 Clinical Features")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# If there's a group field, visualize mean age by group
if group_field_id:
    plt.figure(figsize=(6,4))
    sns.barplot(x=group_field_id, y=numeric_field_id, data=clinical_df)
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
This notebook demonstrated:
- Loading the FAIR^2 dataset and its metadata with `mlcroissant`
- Listing all record sets and fields using their `@id`
- Extracting and analyzing tabular data from each record set
- Referencing all entities by their unique `@id` to ensure robust data handling
- Applying basic EDA (filtering, normalization, grouping)
- Visualizing data distributions using Python standard libraries

**Note:** Due to small sample size (N=3), results and visualizations are purely illustrative and not suitable for robust statistical inference.